In [1]:
import random
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, confusion_matrix

seed = 42
random.seed(seed)
torch.manual_seed(seed)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded model: textattack/distilbert-base-uncased-MRPC
Number of labels: 2


In [3]:
target_per_class = 64
dataset_full = load_dataset("glue", "mrpc", split="validation")

label_map = {0: "not_paraphrase", 1: "paraphrase"}
indices_by_class = {0: [], 1: []}

for idx, row in enumerate(dataset_full):
    indices_by_class[row["label"]].append(idx)

available_per_class = {k: len(v) for k, v in indices_by_class.items()}
n_per_class = min(target_per_class, available_per_class[0], available_per_class[1])

selected_indices = indices_by_class[0][:n_per_class] + indices_by_class[1][:n_per_class]
selected_indices = sorted(selected_indices)
dataset = dataset_full.select(selected_indices)

class_counts = {0: 0, 1: 0}
for y in dataset["label"]:
    class_counts[y] += 1

print("Dataset split: glue/mrpc validation (balanced subset)")
print(f"Requested per class: {target_per_class}")
print(f"Using per class: {n_per_class}")
print(f"Total examples: {len(dataset)}")
print(f"Class counts: {class_counts}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation (balanced subset)
Requested per class: 64
Using per class: 64
Total examples: 128
Class counts: {0: 64, 1: 64}
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []
predicted_positive_prob = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    predicted_positive_prob.extend(probs[:, 1].cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")

Completed inference for 128 examples.


In [5]:
accuracy = accuracy_score(labels, predictions)
balanced_acc = balanced_accuracy_score(labels, predictions)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    labels, predictions, average="macro", zero_division=0
)
cm = confusion_matrix(labels, predictions, labels=[0, 1])
per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
    labels, predictions, labels=[0, 1], average=None, zero_division=0
)

print("Evaluation metrics on balanced MRPC validation subset:")
print(f"Accuracy         : {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro Precision  : {macro_precision:.4f}")
print(f"Macro Recall     : {macro_recall:.4f}")
print(f"Macro F1         : {macro_f1:.4f}")
print("Confusion matrix:")
print(cm)

print("Per-class metrics:")
for idx, label_id in enumerate([0, 1]):
    print(
        f"class={label_id} ({label_map[label_id]}), "
        f"precision={per_class_precision[idx]:.4f}, "
        f"recall={per_class_recall[idx]:.4f}, "
        f"f1={per_class_f1[idx]:.4f}, "
        f"support={per_class_support[idx]}"
    )

Evaluation metrics on balanced MRPC validation subset:
Accuracy         : 0.7969
Balanced Accuracy: 0.7969
Macro Precision  : 0.8455
Macro Recall     : 0.7969
Macro F1         : 0.7895
Confusion matrix:
[[39 25]
 [ 1 63]]
Per-class metrics:
class=0 (not_paraphrase), precision=0.9750, recall=0.6094, f1=0.7500, support=64
class=1 (paraphrase), precision=0.7159, recall=0.9844, f1=0.8289, support=64


In [6]:
true_counts = {0: 0, 1: 0}
pred_counts = {0: 0, 1: 0}

for y in labels:
    true_counts[y] += 1
for yhat in predictions:
    pred_counts[yhat] += 1

is_balanced_subset = true_counts[0] == true_counts[1]

print("Class-count verification:")
print(f"True label counts      : {true_counts}")
print(f"Predicted label counts : {pred_counts}")
print(f"Balanced subset check  : {is_balanced_subset}")

Class-count verification:
True label counts      : {0: 64, 1: 64}
Predicted label counts : {0: 40, 1: 88}
Balanced subset check  : True


In [7]:
false_positives = []
false_negatives = []

for i in range(len(dataset)):
    row = dataset[i]
    record = {
        "index": i,
        "true_label": labels[i],
        "pred_label": predictions[i],
        "confidence": confidences[i],
        "p_paraphrase": predicted_positive_prob[i],
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
    }
    if labels[i] == 0 and predictions[i] == 1:
        false_positives.append(record)
    elif labels[i] == 1 and predictions[i] == 0:
        false_negatives.append(record)

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

def show_errors(title, rows, limit=5):
    print(title)
    if not rows:
        print("None")
        return
    print("idx | true | pred | conf | p_paraphrase | sentence1 | sentence2")
    for r in rows[:limit]:
        s1 = r["sentence1"].replace("\n", " ")[:70]
        s2 = r["sentence2"].replace("\n", " ")[:70]
        print(
            f"{r['index']:>3} | {r['true_label']} | {r['pred_label']} | "
            f"{r['confidence']:.4f} | {r['p_paraphrase']:.4f} | {s1} | {s2}"
        )

show_errors("Compact false positive table (first 5)", false_positives, limit=5)
print("-" * 120)
show_errors("Compact false negative table (first 5)", false_negatives, limit=5)

False positives: 25
False negatives: 1
Compact false positive table (first 5)
idx | true | pred | conf | p_paraphrase | sentence1 | sentence2
  6 | 0 | 1 | 0.9295 | 0.9295 | While dioxin levels in the environment were up last year , they have d | The Institute said dioxin levels in the environment have fallen by as 
 26 | 0 | 1 | 0.8425 | 0.8425 | Cooley said he expects Muhammad will similarly be called as a witness  | Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hea
 35 | 0 | 1 | 0.9618 | 0.9618 | Bush wanted " to see an aircraft landing the same way that the pilots  | On Tuesday , before Byrd 's speech , Fleischer said Bush wanted ' ' to
 60 | 0 | 1 | 0.9689 | 0.9689 | Terri Schiavo , 39 , is expected to die sometime in the next two weeks | Terri Schiavo , 39 , underwent the procedure at the Tampa Bay area hos
 76 | 0 | 1 | 0.9726 | 0.9726 | McCabe said he was considered a witness , not a suspect . | " He is not considered a suspect , " McCabe said .
-----------

In [8]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation (balanced subset)")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"samples_per_class={n_per_class}")
print(f"true_count_not_paraphrase={true_counts[0]}")
print(f"true_count_paraphrase={true_counts[1]}")
print(f"balanced_subset={is_balanced_subset}")
print(f"accuracy={accuracy:.4f}")
print(f"balanced_accuracy={balanced_acc:.4f}")
print(f"macro_precision={macro_precision:.4f}")
print(f"macro_recall={macro_recall:.4f}")
print(f"macro_f1={macro_f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print(f"precision_not_paraphrase={per_class_precision[0]:.4f}")
print(f"recall_not_paraphrase={per_class_recall[0]:.4f}")
print(f"f1_not_paraphrase={per_class_f1[0]:.4f}")
print(f"support_not_paraphrase={int(per_class_support[0])}")
print(f"precision_paraphrase={per_class_precision[1]:.4f}")
print(f"recall_paraphrase={per_class_recall[1]:.4f}")
print(f"f1_paraphrase={per_class_f1[1]:.4f}")
print(f"support_paraphrase={int(per_class_support[1])}")
print(f"pred_count_not_paraphrase={pred_counts[0]}")
print(f"pred_count_paraphrase={pred_counts[1]}")
print(f"false_positives={len(false_positives)}")
print(f"false_negatives={len(false_negatives)}")

RESULT SUMMARY
model=textattack/distilbert-base-uncased-MRPC
dataset_split=glue/mrpc validation (balanced subset)
device=mps
num_examples=128
samples_per_class=64
true_count_not_paraphrase=64
true_count_paraphrase=64
balanced_subset=True
accuracy=0.7969
balanced_accuracy=0.7969
macro_precision=0.8455
macro_recall=0.7969
macro_f1=0.7895
confusion_matrix=[[39, 25], [1, 63]]
precision_not_paraphrase=0.9750
recall_not_paraphrase=0.6094
f1_not_paraphrase=0.7500
support_not_paraphrase=64
precision_paraphrase=0.7159
recall_paraphrase=0.9844
f1_paraphrase=0.8289
support_paraphrase=64
pred_count_not_paraphrase=40
pred_count_paraphrase=88
false_positives=25
false_negatives=1
